# SUTV: Salon & Barbershop Density by City / Neighborhood / Zip Code

**Project:** Shape Up The Vote — 2024 Voter Engagement Initiative  
**Author:** Stann  
**Last Updated:** 2024

---

This notebook processes and analyzes data for voter engagement efforts in major cities within swing states in the USA, with a focus on barbershops and salons.

The **primary goal** is to identify high-priority areas within these cities by:
1. Examining postal codes, shop locations, and demographic information
2. Building state-specific salon density benchmarks
3. Creating a normalized **Service Density KPI** to assess how well each locality is covered in our scraped dataset

> **Note (Portfolio Version):** The original notebook ran SQL queries against a live BigQuery database (`sutv.*` tables) and used the Hex notebook platform. This version uses synthetic data that mirrors the real schema so the full analysis pipeline can be run locally, end-to-end.


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import warnings
warnings.simplefilter('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded.")


## 2. State-Level Salon & Barbershop Counts

The table below shows the estimated total number of licensed salons and barbershops per target state, sourced from [SalonSpaConnection.com Hair Salon Industry Statistics (2023)](https://salonspaconnection.com/beauty-hair-salon-industry-statistics-in-2023/).

> **Researcher's Note (Stann):** Some states were uncooperative or had inaccessible information, which specifically affected estimates for New Hampshire. Additionally, this analysis primarily focuses on *registered* salons and barbershops. It's important to note that in many communities — particularly communities of color and working-class areas — there is a significant likelihood that hair stylists and beauty professionals may not have a license or operate from an official storefront. This means our scraped dataset likely *under-counts* available engagement points in the communities we care most about.


In [ ]:
# State-level totals from SalonSpaConnection.com (2023)
# Population figures from US Census Bureau (2023 estimates)

state_stats = pd.DataFrame([
    ("AZ", "Arizona",       8_875,  7_431_344),
    ("GA", "Georgia",      13_656, 11_029_227),
    ("NV", "Nevada",        3_422,  3_194_176),
    ("MI", "Michigan",     11_492, 10_037_261),
    ("NC", "North Carolina",16_200, 10_835_491),
    ("PA", "Pennsylvania", 20_322, 12_961_683),
    ("WI", "Wisconsin",     8_637,  5_892_539),
    ("OH", "Ohio",         15_647, 11_785_935),
    ("CO", "Colorado",      6_661,  5_839_926),
    ("FL", "Florida",      40_060, 22_244_823),
    ("TX", "Texas",        32_578, 30_029_572),
    ("VA", "Virginia",      7_630,  8_683_619),
    ("NH", "New Hampshire", 1_625,  1_395_231),
], columns=["state", "state_name", "total_salons", "population_2023"])

state_stats["ratio_people_per_salon"] = (
    state_stats["population_2023"] / state_stats["total_salons"]
).round(0).astype(int)

state_stats["vs_national_avg"] = state_stats["ratio_people_per_salon"].apply(
    lambda r: "More dense" if r < 773 else "Less dense"
)

NATIONAL_TOTAL_SALONS = 444_102
NATIONAL_POPULATION   = 343_477_335
NATIONAL_BENCHMARK    = NATIONAL_POPULATION / NATIONAL_TOTAL_SALONS  # ~773

print(f"National benchmark: 1 salon per {NATIONAL_BENCHMARK:.0f} people")
print(f"Est. total salons across 13 swing states: {state_stats['total_salons'].sum():,}\n")
state_stats


## 3. Density Benchmark Methodology

### US Industry Benchmark

A common industry benchmark in the US is approximately **1 salon/barbershop per 1,500–2,000 people**. However, this is too broad for city and district-level analysis because it undercounts barbershops per capita in urban areas due to the weighting of rural/suburban localities.

### National Average Benchmark

Instead of the industry benchmark, we use the **national average benchmark (1:773)** derived from SalonSpaConnection.com data. This is more accurate for urban areas.

### State-Specific Benchmark

For city-level analysis, we go one step further: we calculate a **state-specific benchmark** that accounts for regional density trends, which the national average obscures.

**Example — Pennsylvania:**

| Step | Calculation | Result |
|------|-------------|--------|
| State salon density | 20,322 salons ÷ 12,961,683 people | **1:638** |
| Philadelphia Urban Area (PUA) population share | 5,700,000 ÷ 12,800,000 | **44.5%** |
| Estimated salons in PUA | 44.5% × 20,322 | **~9,043** |

**Takeaway:** Instead of the static national benchmark of 1:773, we use state-average benchmarks that are more accurate for each state's density distribution.


In [ ]:
# Compute state-specific benchmarks
state_stats["state_benchmark"] = state_stats["ratio_people_per_salon"].round(0).astype(int)

# Visualize state benchmarks vs national average
fig, ax = plt.subplots(figsize=(11, 5))

colors = ["#54A24B" if r < NATIONAL_BENCHMARK else "#E45756"
          for r in state_stats["ratio_people_per_salon"]]
bars = ax.bar(state_stats["state"], state_stats["ratio_people_per_salon"], color=colors)

ax.axhline(NATIONAL_BENCHMARK, color='#4C78A8', linestyle='--', linewidth=1.5,
           label=f'National avg (1:{NATIONAL_BENCHMARK:.0f})')

ax.set_ylabel("People per Salon/Barbershop", fontsize=11)
ax.set_title("State Salon Density vs. National Average\n(lower = more salons per capita)", 
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

for bar, val in zip(bars, state_stats["ratio_people_per_salon"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'1:{val}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig("state_density_benchmarks.png", dpi=150, bbox_inches='tight')
plt.show()


## 4. Service Density KPI

To understand district-level trends in salon density, we create a normalized KPI between **0 and 1** that measures the density of salons/barbershops in a given locality *relative to the state benchmark*.

**Formula:**

```
KPI = state_benchmark_ratio / actual_people_per_salon
    = capped at 1.0 (≥1.0 means coverage is adequate)
```

| KPI Value | Interpretation |
|-----------|----------------|
| **< 0.5** | Severely under-scraped — needs more data collection |
| **0.5–0.8** | Under-scraped — coverage gaps likely |
| **0.8–1.0** | Near benchmark — minor gaps |
| **≥ 1.0** | Adequate coverage — meets or exceeds state benchmark |


In [ ]:
def calc_service_density_kpi(actual_people_per_salon: float, state_benchmark: float) -> float:
    """
    Normalized KPI measuring how well a locality's scraped shop count
    meets the state-level density benchmark.
    
    KPI >= 1.0  → coverage is adequate (meets or exceeds benchmark)
    KPI < 1.0   → under-scraped relative to benchmark
    """
    if actual_people_per_salon == 0:
        return 0.0
    return min(state_benchmark / actual_people_per_salon, 1.0)


# ── Synthetic locality-level shop counts ─────────────────────────────────────
# Mirrors the result of the SQL query that joined sutv_leads + raw_zips per city

locality_data = [
    # Philadelphia districts
    ("Philadelphia", "19121", "PA", 28000, 45),
    ("Philadelphia", "19132", "PA", 22000, 28),
    ("Philadelphia", "19140", "PA", 36000, 52),
    ("Philadelphia", "19143", "PA", 41000, 61),
    ("Philadelphia", "19103", "PA", 33000, 87),
    # Atlanta districts
    ("Atlanta",      "30301", "GA", 42000, 95),
    ("Atlanta",      "30303", "GA", 18000, 30),
    ("Atlanta",      "30310", "GA", 25000, 38),
    ("Atlanta",      "30318", "GA", 31000, 55),
    # Detroit districts
    ("Detroit",      "48201", "MI", 15000, 18),
    ("Detroit",      "48205", "MI", 24000, 22),
    ("Detroit",      "48213", "MI", 27000, 25),
    ("Detroit",      "48227", "MI", 31000, 29),
    # Charlotte districts
    ("Charlotte",    "28206", "NC", 29000, 47),
    ("Charlotte",    "28208", "NC", 33000, 55),
    ("Charlotte",    "28212", "NC", 37000, 60),
    # Columbus districts
    ("Columbus",     "43205", "OH", 19000, 21),
    ("Columbus",     "43211", "OH", 27000, 30),
]

localities = pd.DataFrame(locality_data,
    columns=["city", "zip_code", "state", "population", "scraped_shops"])

# Join state benchmark
localities = localities.merge(
    state_stats[["state", "state_benchmark"]],
    on="state", how="left"
)

# Compute actual density and KPI
localities["people_per_scraped_shop"] = (
    localities["population"] / localities["scraped_shops"]
).round(1)

localities["service_density_kpi"] = localities.apply(
    lambda row: calc_service_density_kpi(
        row["people_per_scraped_shop"], row["state_benchmark"]
    ), axis=1
).round(3)

localities["coverage_status"] = localities["service_density_kpi"].apply(
    lambda k: "✅ Adequate"   if k >= 1.0 else
              "🟡 Near"       if k >= 0.8 else
              "🟠 Under"      if k >= 0.5 else
              "🔴 Severely Under"
)

localities.sort_values(["city", "service_density_kpi"]).reset_index(drop=True)


## 5. City Deep-Dives

For each major city, we plot the Service Density KPI by zip code to identify which neighborhoods need additional data scraping.

> **Researcher's Note (Stann):** This analysis focuses on *urban areas* within the most populous cities in each state, targeting the main city and its surrounding suburbs. It is important to distinguish these from larger regions like Greater Metropolitan Statistical Areas (MSAs), which can include suburbs from neighboring states and would skew our state-specific benchmarks.


In [ ]:
cities = localities["city"].unique()
n_cities = len(cities)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, city in zip(axes, cities):
    city_df = localities[localities["city"] == city].sort_values("service_density_kpi")
    
    colors = [
        "#54A24B" if k >= 1.0 else
        "#EECA3B" if k >= 0.8 else
        "#F58518" if k >= 0.5 else
        "#E45756"
        for k in city_df["service_density_kpi"]
    ]
    
    bars = ax.barh(city_df["zip_code"], city_df["service_density_kpi"], color=colors)
    ax.axvline(1.0, color='gray', linestyle='--', linewidth=1, label='Benchmark (1.0)')
    ax.set_xlim(0, 1.15)
    ax.set_title(f"{city}", fontsize=11, fontweight='bold')
    ax.set_xlabel("Service Density KPI", fontsize=9)
    ax.legend(fontsize=8)
    
    for bar, val in zip(bars, city_df["service_density_kpi"]):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.2f}', va='center', fontsize=8)

# Hide unused subplots
for ax in axes[n_cities:]:
    ax.set_visible(False)

fig.suptitle("Service Density KPI by Zip Code\n(≥1.0 = adequate scraping coverage)", 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("service_density_kpi_by_city.png", dpi=150, bbox_inches='tight')
plt.show()


## 6. Summary — Coverage Gaps by City

In [ ]:
summary = (
    localities.groupby(["city", "state"])
    .agg(
        total_zips       = ("zip_code", "count"),
        avg_kpi          = ("service_density_kpi", "mean"),
        under_scraped    = ("coverage_status", lambda x: (x.str.contains("Under|Severely")).sum()),
        adequate         = ("coverage_status", lambda x: (x == "✅ Adequate").sum()),
    )
    .round(3)
    .sort_values("avg_kpi")
    .reset_index()
)

print("Cities with lowest average coverage (most in need of additional scraping):")
summary


---

## Appendix: State Benchmark Calculations

Each state benchmark is derived using three steps:

1. **Calculate state-wide salon density** → `state_population / total_salons`
2. **Determine the urban area's population share** → `urban_pop / state_pop`
3. **Estimate expected salons in the urban area** → `population_share × total_salons`

This produces a more accurate density expectation than the static national benchmark (1:773), because it accounts for the fact that urban areas concentrate both population and salons differently across states.

See the full state-by-state calculations in the markdown cells of the original `salonbarbershop_density_by_cityneighborhoodzipcode.ipynb`.
